In [2]:
import numpy as np
import pandas as pd
n = 1000
m = 500
rho = 0.5
maf = 0.2
s2_gxg = 0.3
s2_e = 0.7
Nmc = 100
reps = 1

p = m * (m - 1) // 2
rng = np.random.default_rng()

# ---- genotypes: fixed across replicates ----
causal1 = rng.binomial(1, maf, size=(n, 1))
h1 = rng.binomial(1, maf, size=(n, m))
c1 = rng.binomial(1, rho, size=(n, m))
h1 = np.where(c1, causal1, h1)

causal2 = rng.binomial(1, maf, size=(n, 1))
h2 = rng.binomial(1, maf, size=(n, m))
c2 = rng.binomial(1, rho, size=(n, m))
h2 = np.where(c2, causal2, h2)

X = h1 + h2
Z = (X - X.mean(axis=0)) / X.std(axis=0)

j, k = np.triu_indices(m, k=1)
H = Z[:, j] * Z[:, k]
W = H @ H.T / p

# hivert std
#W = W / np.diag(W).mean()

# ---- replicates: new phenotype each time ----
est_g = np.zeros(reps)
est_e = np.zeros(reps)
real_g = np.zeros(reps)
real_e = np.zeros(reps)

for i in range(reps):
    gamma = rng.normal(0, np.sqrt(s2_gxg / p), size=p)
    g = H @ gamma
    e = rng.normal(0, np.sqrt(s2_e), size=n)
    y = g + e
    est_g[i], est_e[i], _ = MC_REML(W, y, Nmc=Nmc)
    real_g[i] = g.var()
    real_e[i] = e.var()

print(f"mean diag(W) : {np.diag(W).mean():.4f}")
print(f"{'':14}{'true':>8}{'mean':>10}{'var':>10}{'se of mean':>13}")
for lab, arr, tru in [("s2gxg", est_g, s2_gxg), ("s2e", est_e, s2_e),
                      ("var(g)", real_g, s2_gxg), ("var(e)", real_e, s2_e)]:
    print(f"{lab:14}{tru:8.4f}{arr.mean():10.4f}{arr.var(ddof=1):10.4f}"
          f"{arr.std(ddof=1)/np.sqrt(len(arr)):13.4f}")

mean diag(W) : 1.3352
                  true      mean       var   se of mean
s2gxg           0.3000    0.3049       nan          nan
s2e             0.7000    0.6679       nan          nan
var(g)          0.3000    0.3855       nan          nan
var(e)          0.7000    0.7161       nan          nan


C:\Users\Ziyan Zhang\AppData\Local\Temp\ipykernel_42188\3628713542.py:55: RuntimeWarning: Degrees of freedom <= 0 for slice
  print(f"{lab:14}{tru:8.4f}{arr.mean():10.4f}{arr.var(ddof=1):10.4f}"
D:\anaconda\Lib\site-packages\numpy\_core\_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
D:\anaconda\Lib\site-packages\numpy\_core\_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [116]:
# V_ell = V_gamma * c:  check the O(nm) closed form for c against the exact value
import numpy as np

n = 10000
m = 100
rho = 0.5
maf = 0.1
s2_gxg = 0.3
reps = 500

p = m * (m - 1) // 2
rng = np.random.default_rng()

# ---- genotypes: every SNP tags one causal haplotype w.p. rho ----
causal1 = rng.binomial(1, maf, size=(n, 1))
h1 = rng.binomial(1, maf, size=(n, m))
c1 = rng.binomial(1, rho, size=(n, m))
h1 = np.where(c1, causal1, h1)

causal2 = rng.binomial(1, maf, size=(n, 1))
h2 = rng.binomial(1, maf, size=(n, m))
c2 = rng.binomial(1, rho, size=(n, m))
h2 = np.where(c2, causal2, h2)

X = h1 + h2
Z = (X - X.mean(axis=0)) / X.std(axis=0)

j, k = np.triu_indices(m, k=1)
H = Z[:, j] * Z[:, k]

# ---- c from the skewness closed form: O(nm), no m x m, no H ----
p_af = X.mean(axis=0) / 2
s = (1 - 2 * p_af) / np.sqrt(2 * p_af * (1 - p_af))
c_hat = 1 + (np.linalg.norm(Z @ s) ** 2 / n - (s ** 2).sum()) / (2 * p)

# ---- exact c: mean sample variance of the p product columns ----
c_exact = H.var(axis=0).mean()

# ---- realized variance of g, over many draws of gamma ----
B = rng.normal(0, np.sqrt(s2_gxg / p), size=(p, reps))
vl = (H @ B).var(axis=0)
sd1 = vl.std(ddof=1)

err = abs(s2_gxg * (c_hat - c_exact))

print(f"n = {n}, m = {m}, rho = {rho}")
print(f"c hat  (closed form)     : {c_hat:.4f}")
print(f"c exact (col variances)  : {c_exact:.4f}")
print()
print(f"V_gamma                  : {s2_gxg:.4f}")
print(f"V_ell = V_gamma * c_exact  : {s2_gxg * c_exact:.4f}")
print(f"V_ell realized           : {vl.mean():.4f} +- {sd1 / np.sqrt(reps):.4f}")
print()
print(f"(A) error from c_hat     : {err:.5f}")
print(f"(B) sd of a single draw  : {sd1:.5f}")
print(f"    A / B                : {err / sd1:.3f}")

n = 10000, m = 100, rho = 0.5
c hat  (closed form)     : 1.8867
c exact (col variances)  : 1.9010

V_gamma                  : 0.3000
V_ell = V_gamma * c_hat  : 0.5660
V_ell realized           : 0.5721 +- 0.0060

(A) error from c_hat     : 0.00430
(B) sd of a single draw  : 0.13474
    A / B                : 0.032


In [18]:
# decide r
X = pd.read_csv("ContiguousSNP_n1000_m1000.csv", header=None).to_numpy(dtype=np.float64)[:,:100]
s = np.linalg.svd(X, compute_uv=False)
lam = s ** 2
rho_cum = np.cumsum(lam) / lam.sum()

for target in [0.90, 0.95, 0.99, 0.999]:
    r = np.searchsorted(rho_cum, target) + 1
    print(f"{target:6.3f}  ->  r = {r:5d}   ({r / len(lam):.1%} of rank)")

 0.900  ->  r =     5   (5.0% of rank)
 0.950  ->  r =     7   (7.0% of rank)
 0.990  ->  r =    18   (18.0% of rank)
 0.999  ->  r =    35   (35.0% of rank)


In [3]:
# ----------------------------------------------------------------- MC AI-REML
def _v_matvec(W, s2gxg, s2e, B):
    """Apply V = s2gxg W + s2e I to B using the pre-computed dense W."""
    return s2gxg * (W @ B) + s2e * B


def _cg_batched(matvec, Bmat, x0=None, tol=1e-6, maxiter=1000):
    """Conjugate gradient for the SPD system V X = Bmat, all columns together."""
    n_, c = Bmat.shape
    X = np.zeros((n_, c)) if x0 is None else x0.copy()
    R = Bmat - matvec(X)
    P = R.copy()
    rs_old = np.sum(R * R, axis=0)
    b_norm = np.sqrt(np.sum(Bmat * Bmat, axis=0))
    b_norm[b_norm == 0.0] = 1.0

    for _ in range(maxiter):
        VP = matvec(P)
        alpha = rs_old / np.sum(P * VP, axis=0)
        X += alpha * P
        R -= alpha * VP
        rs_new = np.sum(R * R, axis=0)
        if np.max(np.sqrt(rs_new) / b_norm) < tol:
            break
        beta = rs_new / rs_old
        P = R + beta * P
        rs_old = rs_new
    return X


def mc_reml(W, y, iters=30, Nmc=50, cg_tol=1e-6, cg_maxiter=1000,
            jitter=1e-8, tol=1e-8, lm=1e-3, step_frac=0.5, upper_mult=5.0,
            seed=None, verbose=False):
    """Monte-Carlo AI-REML for V = s2gxg W + s2e I (dense pre-computed W)."""
    y = np.asarray(y, dtype=float).flatten()
    W = np.asarray(W, dtype=float)
    n_ = y.shape[0]
    k = 2

    vary = y.var()
    s_upper = upper_mult * vary

    rng_ = np.random.default_rng(seed)
    U = rng_.choice([-1.0, 1.0], size=(n_, Nmc))

    s = np.full(k, vary / k)
    AI = np.eye(k)

    yc = y.reshape(n_, 1)
    xbuf = None
    P = None
    G = None

    WU = W @ U

    for it in range(iters):
        s2gxg, s2e = s
        matvec = lambda B: _v_matvec(W, s2gxg, s2e, B)

        xbuf = _cg_batched(matvec, yc, x0=xbuf, tol=cg_tol, maxiter=cg_maxiter)
        x = xbuf[:, 0]

        Wx = W @ x
        xWx = x @ Wx
        xIx = x @ x

        P = _cg_batched(matvec, U, x0=P, tol=cg_tol, maxiter=cg_maxiter)
        trV1W = np.mean(np.sum(P * WU, axis=0))
        trV1I = np.mean(np.sum(P * U, axis=0))

        score = np.array([0.5 * (xWx - trV1W),
                          0.5 * (xIx - trV1I)])

        KX = np.column_stack([Wx, x])
        G = _cg_batched(matvec, KX, x0=G, tol=cg_tol, maxiter=cg_maxiter)
        AI = 0.5 * (KX.T @ G)
        AI = 0.5 * (AI + AI.T)

        dA = np.abs(np.diag(AI))
        ridge = lm * (dA.mean() + 1e-12)
        step = np.linalg.solve(AI + (ridge + jitter) * np.eye(k), score)

        mx = np.abs(step).max()
        max_step = step_frac * vary
        if mx > max_step:
            step *= max_step / mx
        s = np.clip(s + step, 1e-9, s_upper)

        if verbose:
            print(f"iter {it:2d}  s={s}  max|step|={np.abs(step).max():.3e}")
        if np.abs(step).max() < tol:
            break

    return s, AI


def MC_REML(W, y, iters=30, Nmc=50, cg_tol=1e-6, cg_maxiter=1000, seed=None):
    """Wrapper: returns (s2gxg_hat, s2e_hat, AI)."""
    s, AI = mc_reml(W, y, iters=iters, Nmc=Nmc, cg_tol=cg_tol,
                    cg_maxiter=cg_maxiter, seed=seed)
    s2gxg_hat, s2e_hat = s
    return s2gxg_hat, s2e_hat, AI


# ------------------------------------------------------------------ simulation
# ---- genotypes: every SNP copies the causal haplotype w.p. rho ----
causal1 = rng.binomial(1, maf, size=(n, 1))
h1 = rng.binomial(1, maf, size=(n, m))
c1 = rng.binomial(1, rho, size=(n, m))
h1 = np.where(c1, causal1, h1)

causal2 = rng.binomial(1, maf, size=(n, 1))
h2 = rng.binomial(1, maf, size=(n, m))
c2 = rng.binomial(1, rho, size=(n, m))
h2 = np.where(c2, causal2, h2)

X = h1 + h2                                       # 0 / 1 / 2

# ---- standardization ----
Z = (X - X.mean(axis=0)) / X.std(axis=0)

# ---- epistatic design matrix and GRM ----
j, k = np.triu_indices(m, k=1)
H = Z[:, j] * Z[:, k]                             # n x p
W = H @ H.T / p                                   # n x n

# ---- phenotype ----
gamma = rng.normal(0, np.sqrt(s2_gxg / p), size=p)
g = H @ gamma
e = rng.normal(0, np.sqrt(s2_e), size=n)
y = g + e

# ---- estimation ----
s2gxg_hat, s2e_hat, AI = MC_REML(W, y, Nmc=Nmc)
se = np.sqrt(np.diag(np.linalg.inv(AI)))

print(f"mean diag(W) : {np.diag(W).mean():.4f}")
print(f"true         : s2gxg = {s2_gxg:.4f}   s2e = {s2_e:.4f}")
print(f"realized     : var(g) = {g.var():.4f}   var(e) = {e.var():.4f}")
print(f"MC AI-REML   : s2gxg = {s2gxg_hat:.4f} +- {se[0]:.4f}"
      f"   s2e = {s2e_hat:.4f} +- {se[1]:.4f}")

mean diag(W) : 1.3438
true         : s2gxg = 0.3000   s2e = 0.7000
realized     : var(g) = 0.3804   var(e) = 0.6985
MC AI-REML   : s2gxg = 0.3242 +- 0.0615   s2e = 0.7096 +- 0.0576


In [9]:
import numpy as np
from scipy.sparse.linalg import svds


def variance_explained(Z, r):
    """Cumulative share of genotypic variance carried by the leading r PCs.

    rho(r) = sum_{s<=r} lam_s / sum_s lam_s,  lam_s the eigenvalues of Z Z'.

    The denominator is tr(Z'Z) = ||Z||_F^2 and needs no factorization, so only
    the leading r singular values are computed.  Returns a scalar.
    """
    full = min(Z.shape)
    r = min(r, full)
    if r >= full - 1:                       # ARPACK needs k < min(n, m)
        s = np.linalg.svd(Z, compute_uv=False)[:r]
    else:
        s = svds(Z, k=r, return_singular_vectors=False, v0=np.ones(full))
    return (s ** 2).sum() / np.einsum('ij,ij->', Z, Z)

np.float64(1.0000000000000022)